# Evaluación del agente

Estudiamos la búsqueda de evidencia y evaluamos las 20 preguntas del equipo y las 20 oficiales.
**Run All ejecuta el flujo completo** y reanuda los checkpoints compatibles.
Protocolo de evaluación 5; la ejecución anterior se conserva en el [historial](README.md#resultados-historicos).


## 0 · Configuración

DeepSeek V4 Flash para el agente; Voyage 4 Lite para embeddings cloud.
Evaluación local del framework, sin modelo juez.
Seleccionamos el modo de búsqueda en §3; el agente devuelve contexto por sección.
Antes de ejecutar: entorno del proyecto, `OPENROUTER_API_KEY` externa y Docker en marcha.
Las llamadas nuevas tienen coste. [Instalación y credenciales](README.md#entorno).


In [1]:
import os, sys, json, hashlib, time, urllib.request
from pathlib import Path
from datetime import datetime, timezone
from contextlib import nullcontext
inicio = Path.cwd().resolve()
RAIZ = next(p for p in (inicio, *inicio.parents)
            if (p / "src/taller_nlp").is_dir() and (p / "experimentos/baseline.py").is_file())
for ruta in (RAIZ, RAIZ / "src"):
    sys.path.insert(0, str(ruta))

# Privacidad: no enviar preguntas, respuestas ni trazas a LangSmith.
os.environ.update({"LANGSMITH_TRACING": "false", "LANGCHAIN_TRACING_V2": "false"})

import numpy as np
import pandas as pd
from IPython.display import display
import miax_s2
from experimentos import baseline
from experimentos.jchulvi import agente, qdrant_local
from taller_nlp import (CasoGolden, FabricaHerramientas, ManifiestoExperimento,
                        RespuestaAgente, RespuestaFinanciera)
from taller_nlp.contracts import ResultadoPregunta, VERSION_PROTOCOLO_CITAS
from taller_nlp.hashing import calcular_sha256
from taller_nlp.model_resilience import es_error_transitorio_modelo

assert Path(agente.__file__).resolve().is_relative_to(RAIZ)
DIRECTORIO = RAIZ / "experimentos/jchulvi"
ESTUDIO = agente.DIRECTORIO_RESULTADOS
ESTUDIO.mkdir(parents=True, exist_ok=True)
# §3 selecciona el modo por recall@5 antes de evaluar las 40 preguntas.
MODELO_AGENTE = agente.MODELO_CLOUD
MODO_RETRIEVAL = "denso"
MODELO_EMBEDDINGS = agente.MODELO_EMBEDDINGS

assert MODO_RETRIEVAL in {"bm25", "denso", "hibrido"}
assert os.getenv("OPENROUTER_API_KEY"), "Configura OPENROUTER_API_KEY fuera del notebook antes de iniciar el kernel."
print("Worktree:", RAIZ)
print("Agente: agente.py | modelo:", MODELO_AGENTE)
print("Evaluador local · protocolo de citas:", VERSION_PROTOCOLO_CITAS)
print("Recuperación del agente:", MODO_RETRIEVAL, "| contexto: sección")
print("Máximo de llamadas al modelo por pregunta:", agente.MAX_LLAMADAS_MODELO)
print("Ejecución: comprobaciones, estudio de retrieval y 20 + 20 preguntas.")
print("Proveedor y techos de precio (USD/M tokens):", agente.PROVEEDOR_CLOUD)

def leer_cuenta():
    # Solo metadatos de gasto; nunca se imprimen ni guardan credenciales.
    peticion = urllib.request.Request(
        "https://openrouter.ai/api/v1/key",
        headers={"Authorization": "Bearer " + os.environ["OPENROUTER_API_KEY"]})
    with urllib.request.urlopen(peticion, timeout=30) as r:
        datos = json.load(r)["data"]
    return {"utc": datetime.now(timezone.utc).isoformat(),
            **{k: datos.get(k) for k in ("usage", "limit", "limit_remaining")}}

CUENTA_INICIAL = leer_cuenta()
print("Límite existente de la clave y margen (USD):",
      CUENTA_INICIAL["limit"], CUENTA_INICIAL["limit_remaining"])


Worktree: /Users/jchulvi/projects/Taller_NLP-jchulvi-notebook
Agente: agente.py | modelo: openrouter:deepseek/deepseek-v4-flash-0731
Evaluador local · protocolo de citas: 5
Recuperación del agente: denso | contexto: sección
Máximo de llamadas al modelo por pregunta: 18
Ejecución: comprobaciones, estudio de retrieval y 20 + 20 preguntas.
Proveedor y techos de precio (USD/M tokens): {'only': ['deepinfra'], 'require_parameters': True, 'max_price': {'prompt': 0.1, 'completion': 0.2}}
Límite existente de la clave y margen (USD): 5 4.033977984


## 1 · Datos y herramientas

Comprobamos el corpus, los dos conjuntos de 20 preguntas y las cuatro herramientas del agente.


In [2]:
corpus = baseline.crear_corpus_baseline()
corpus_agente = agente.crear_corpus("seccion")
DATASETS = {"propio": RAIZ / "golden_set.jsonl",
            "oficial": RAIZ / "golden_set_oficial.jsonl"}
HASH_AGENTE = calcular_sha256(Path(agente.__file__))
HASH_BASELINE = calcular_sha256(Path(baseline.__file__))
HASH_QDRANT = calcular_sha256(Path(qdrant_local.__file__))
HASH_GOLDEN = {nombre: calcular_sha256(ruta) for nombre, ruta in DATASETS.items()}
CASOS = {nombre: CasoGolden.cargar_jsonl(ruta, corpus, numero_esperado=20, minimo_comparativas=6)
         for nombre, ruta in DATASETS.items()}
GOLDEN = {nombre: [caso.model_dump() for caso in casos] for nombre, casos in CASOS.items()}
display(pd.DataFrame([{
    "conjunto": nombre, "archivo": DATASETS[nombre].name, "preguntas": len(casos),
    **{familia: sum(c.familia == familia for c in casos)
       for familia in ("extractiva", "numerica", "comparativa")},
    "con_ancla": sum(c.ancla_texto is not None for c in casos),
} for nombre, casos in CASOS.items()]))
chunks = pd.read_json(corpus.ruta_chunks, lines=True)
display(chunks.groupby("item").agg(fragmentos=("chunk_id", "size"),
                                  tokens_medios=("n_tokens", "mean"),
                                  con_tabla=("contiene_tabla", "sum")).round(1))
assert len(chunks) == 1749 and len(CASOS) == 2
print("§1: corpus y ambos golden sets válidos.")

,conjunto,archivo,preguntas,extractiva,numerica,comparativa,con_ancla
0,propio,golden_set.jsonl,20,6,6,8,14
1,oficial,golden_set_oficial.jsonl,20,6,7,7,13


,fragmentos,tokens_medios,con_tabla
item,,,
1A,533,443.2,34
7,307,371.1,120
7A,37,344.9,18
8,872,389.6,549


§1: corpus y ambos golden sets válidos.


In [3]:
# Comprobación de las cuatro herramientas, sin LLM ni embeddings locales.
recuperador_lexico = agente.RecuperadorHibrido(corpus_agente, modo="bm25")
herramientas = FabricaHerramientas(corpus_agente, retriever=recuperador_lexico).crear().por_nombre
assert set(herramientas) == {"list_available", "get_xbrl_fact", "search_filings", "read_section"}
print(herramientas["list_available"].invoke({}))
print(herramientas["get_xbrl_fact"].invoke(
    {"ticker": "NVDA", "fiscal_year": 2025, "concept": "Revenues"}))
assert {"respuesta", "cifra", "unidad", "ticker", "ejercicio", "fuente", "cita", "chunk_id"} <= set(RespuestaFinanciera.model_fields)

- **AAPL** — Apple Inc. (CIK 320193): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8]
- **MSFT** — MICROSOFT CORP (CIK 789019): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8]
- **AMZN** — AMAZON COM INC (CIK 1018724): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8]
- **NVDA** — NVIDIA CORP (CIK 1045810): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8]
- **META** — Meta Platforms, Inc. (CIK 1326801): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8]
- **GOOGL** — Alphabet Inc. (CIK 1652044): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8]
NVDA FY2025 · Revenues = 130,497,000,000 USD (cierre de ejercicio 2025-01-26, según el 10-K)


## 2 · Guardrail y evaluador

Comprobamos el formato, las cifras XBRL y las citas. El evaluador no usa un modelo juez.
El agente tiene 18 llamadas; la última se reserva para la respuesta estructurada.
No se extraen cifras de la prosa: es una limitación respecto al enunciado.


In [4]:
# Un importe correcto pasa; uno alterado o no consultado vuelve al modelo.
from langchain_core.messages import AIMessage, ToolMessage
from pydantic import ValidationError
hecho = pd.read_parquet(corpus.ruta_xbrl).query(
    "ticker == 'NVDA' and fiscal_year == 2025 and concept == 'Revenues'").iloc[0]
salida = RespuestaFinanciera(respuesta="Dato consultado en XBRL.", cifra=float(hecho.value),
                            unidad=hecho.unit, ticker="NVDA", ejercicio=2025, fuente="xbrl")
mensajes = [AIMessage(content="", tool_calls=[{
    "id": "xbrl", "name": "get_xbrl_fact",
    "args": {"ticker": "NVDA", "fiscal_year": 2025, "concept": "Revenues"},
}]), ToolMessage(content=str(hecho.value), tool_call_id="xbrl", name="get_xbrl_fact")]
guardrail = agente.ValidarEvidencia(corpus_agente)
assert guardrail.after_model({"structured_response": salida, "messages": mensajes}, None) is None
incorrecta = salida.model_copy(update={"cifra": salida.cifra * 1.1})
assert guardrail.after_model({"structured_response": incorrecta, "messages": mensajes}, None)["jump_to"] == "model"
assert guardrail.after_model({"structured_response": salida, "messages": []}, None)["jump_to"] == "model"
# Contratos genéricos: no son reglas para preguntas ni respuestas del golden.
# Pydantic valida campos; ToolStrategy comunica sus errores sin otro bucle propio.
for cambios in ({"cifra": None, "unidad": "USD"}, {"respuesta": "   "}):
    try:
        RespuestaFinanciera.model_validate(salida.model_dump() | cambios)
    except ValidationError:
        pass
    else:
        raise AssertionError("El esquema aceptó campos incoherentes.")
assert guardrail.after_model({"messages": [AIMessage(content="Respuesta sin estructura.")]}, None) is None
assert guardrail.after_model({"messages": [mensajes[0]]}, None) is None  # dejar ejecutar la tool
comparacion = mensajes + [AIMessage(content="", tool_calls=[{
    "id": "anterior", "name": "get_xbrl_fact",
    "args": {"ticker": "NVDA", "fiscal_year": 2024, "concept": "Revenues"},
}]), ToolMessage(content="Hecho consultado", tool_call_id="anterior", name="get_xbrl_fact")]
# Consultar dos ejercicios no obliga a declarar una comparación en la salida.
assert guardrail.after_model({"structured_response": salida, "messages": comparacion}, None) is None
assert agente.MAX_LLAMADAS_MODELO == 18
assert guardrail.after_agent({"structured_response": salida}, None) is None
cierre_fallido = guardrail.after_agent({"structured_response": None}, None)
assert cierre_fallido["messages"][0].content.startswith("No he podido completar")
print("Esquema: campos coherentes. Guardrail: evidencia. Límite 18 y aviso de fallo comprobados.")
print("La clasificación de la pregunta depende del prompt; no hay parser de prosa ni lectura del golden.")

Esquema: campos coherentes. Guardrail: evidencia. Límite 18 y aviso de fallo comprobados.
La clasificación de la pregunta depende del prompt; no hay parser de prosa ni lectura del golden.


In [5]:
from taller_nlp.citas import cita_esta_respaldada, CARACTERES_CITA_COMPROBADOS


def huella_json(datos):
    return hashlib.sha256(json.dumps(datos, sort_keys=True).encode()).hexdigest()


configuracion_evaluador = {"version_protocolo_citas": VERSION_PROTOCOLO_CITAS}
# Comprobaciones locales del contrato; no son una calibración semántica.
texto_control = "Revenue increased during the fiscal year."
assert cita_esta_respaldada(texto_control, texto_control)
assert cita_esta_respaldada("REVENUE  increased", texto_control)
assert not cita_esta_respaldada("Revenue decreased", texto_control)
assert not cita_esta_respaldada("", texto_control)
print(f"Protocolo {VERSION_PROTOCOLO_CITAS}: comprobaciones locales correctas.")
print(f"Citas: se busca el prefijo de {CARACTERES_CITA_COMPROBADOS} caracteres tras normalizar espacios y mayúsculas.")
print("Esto no comprueba el significado de toda la respuesta. Sin llamadas a modelos.")


Protocolo 5: comprobaciones locales correctas.
Citas: se busca el prefijo de 120 caracteres tras normalizar espacios y mayúsculas.
Esto no comprueba el significado de toda la respuesta. Sin llamadas a modelos.


## 3 · Estudio de retrieval

Comparamos BM25 y las cinco variantes de S2 sobre 14 anclas del equipo y 13 oficiales.
`recall@5`: evidencia presente entre los cinco primeros fragmentos.
Elegimos el modo entre BM25, denso e híbrido con los mismos filtros, sin reescritura adicional:
mayor recall conjunto (27 casos); en empate, BM25 → denso → híbrido.
Los filtros del golden solo se usan aquí. Las reescrituras son un diagnóstico aparte;
el agente formula sus propias búsquedas en inglés. No es una prueba hold-out.


In [6]:
# Índice independiente por modelo y contrato; conserva el índice anterior.
RUTA_CLOUD = agente.preparar_embeddings_cloud(modelo=MODELO_EMBEDDINGS)
print("Índice cloud:", RUTA_CLOUD.name)


Índice cloud: embeddings_cloud_92113b6e8fbccc47


### Qdrant

Documentos y consultas se guardan en `embeddings/`, junto a este notebook y disponibles para Git.
Se reutilizan sin API; una consulta nueva usa Voyage.
`qdrant.sesion()` inicia Qdrant, carga el índice y lo detiene al terminar, también ante errores.


In [7]:
qdrant = qdrant_local.QdrantLocal(
    ESTUDIO, ruta_indice=RUTA_CLOUD, corpus=corpus,
    modelo=MODELO_EMBEDDINGS, contrato=agente.CONTRATO_EMBEDDINGS,
)
META_CLOUD = qdrant.metadatos
COLECCION = qdrant.coleccion


In [8]:
# Las cinco configuraciones de S2, más un control léxico sin modelos.
K = 5
PLAN_RETRIEVAL = {
    "0 · BM25 (control)": ("bm25", True, False),
    "1 · denso plano": ("denso", False, False),
    "2 · + metadatos": ("denso", True, False),
    "3 · + híbrido BM25": ("hibrido", True, False),
    "4 · reescritura + denso": ("denso", True, True),
    "5 · reescritura + híbrido": ("hibrido", True, True),
}
PROMPT_REESCRITURA = ("Reescribe esta pregunta como una consulta de búsqueda para un índice "
                     "de informes 10-K en INGLÉS. Conserva empresa y ejercicio; no respondas "
                     "la pregunta. Devuelve SOLO la consulta, sin comillas ni explicación.")
firma_retrieval = {
    "codigo": HASH_AGENTE,
    "qdrant": HASH_QDRANT,
    "metrica": calcular_sha256(RAIZ / "miax_s2.py"),
    "framework": {p.name: calcular_sha256(p) for p in sorted((RAIZ / "src/taller_nlp").glob("*.py"))},
    "modelo_reescritor": MODELO_AGENTE, "prompt": PROMPT_REESCRITURA,
    "modelo_embeddings": MODELO_EMBEDDINGS, "indice": META_CLOUD,
    "chunks": corpus.sha256_chunks, "golden": HASH_GOLDEN,
    "plan": PLAN_RETRIEVAL, "k": K,
}
RUTA_RETRIEVAL = ESTUDIO / ("retrieval_s2_" + huella_json(firma_retrieval)[:16])
RUTA_RETRIEVAL.mkdir(exist_ok=True)
agente.guardar_atomico(RUTA_RETRIEVAL / "configuracion.json",
                          json.dumps(firma_retrieval, indent=2).encode())
reescritor = None
control_reescritor = agente.ControlPeticionesModelo.desde_configuracion(
    baseline.crear_configuracion_baseline().model_copy(update={"max_reintentos_rate_limit": 0}))

def reescribir(pregunta):
    global reescritor
    ruta = RUTA_RETRIEVAL / ("consulta_" + huella_json(pregunta) + ".json")
    if ruta.is_file():
        return json.loads(ruta.read_text())["consulta"]
    if reescritor is None:
        reescritor = agente.crear_modelo_cloud(MODELO_AGENTE, limitador=control_reescritor.limitador)
    inicio = time.perf_counter()
    mensaje = control_reescritor.ejecutar(lambda: reescritor.invoke([
        {"role": "system", "content": PROMPT_REESCRITURA},
        {"role": "user", "content": pregunta},
    ]))
    consulta = mensaje.text.strip()
    if not consulta:
        raise ValueError("Reescritura vacía: no se sustituye por una respuesta preparada.")
    agente.guardar_atomico(ruta, json.dumps({
        "pregunta": pregunta, "consulta": consulta, "modelo": MODELO_AGENTE,
        "segundos": time.perf_counter() - inicio, "tokens": mensaje.usage_metadata,
    }, ensure_ascii=False, indent=2).encode())
    return consulta

def medir_retrieval(dataset, etiqueta, modo, filtros, reescritura):
    ruta = RUTA_RETRIEVAL / f"{dataset}_{etiqueta[0]}.json"
    registros = json.loads(ruta.read_text()) if ruta.is_file() else {}
    casos = [g for g in GOLDEN[dataset] if g["ancla_texto"]]
    if (modo == "bm25" or META_CLOUD is not None) and len(registros) < len(casos):
        retriever = agente.RecuperadorHibrido(
            corpus, modo=modo, modelo_embeddings=MODELO_EMBEDDINGS,
            qdrant_url=qdrant.url if modo != "bm25" else None,
            coleccion=COLECCION if modo != "bm25" else None,
        )
        for g in casos:
            if g["id"] in registros:
                continue
            consulta = reescribir(g["pregunta"]) if reescritura else g["pregunta"]
            parametros = ({"ticker": g["ticker"], "fiscal_year": g["fiscal_year"],
                           "item": g["item_esperado"]} if filtros else {})
            fragmentos = [f.model_dump(mode="json") for f in retriever.buscar(consulta, **parametros, k=K)]
            registros[g["id"]] = {"consulta": consulta, "fragmentos": fragmentos}
            agente.guardar_atomico(ruta, json.dumps(registros, ensure_ascii=False).encode())
    completo = len(registros) == len(casos)
    return {
        "conjunto": dataset, "configuración": etiqueta, "medidas": len(registros), "total": len(casos),
        "recall@5": miax_s2.recall_en_k(casos, {i: r["fragmentos"] for i, r in registros.items()}) if completo else None,
        "reescrituras_por_consulta": int(reescritura), "estado": "completo" if completo else "pendiente",
    }

In [9]:
resultados_retrieval = []
with qdrant.sesion(cargar_indice=False):
    puntos = qdrant.cargar()
    assert puntos == len(chunks) == 1749
    print(f"Qdrant: {puntos} puntos, {META_CLOUD['dimension']} dimensiones; colección {COLECCION}.")
    for dataset in DATASETS:
        for etiqueta, (modo, filtros, reescritura) in PLAN_RETRIEVAL.items():
            fila = medir_retrieval(dataset, etiqueta, modo, filtros, reescritura)
            resultados_retrieval.append(fila)
            print(dataset, etiqueta, f"{fila['medidas']}/{fila['total']}", flush=True)
tabla_retrieval = pd.DataFrame(resultados_retrieval)
display(tabla_retrieval)
tabla_retrieval.to_csv(RUTA_RETRIEVAL / "resumen.csv", index=False)
assert len(tabla_retrieval) == 2 * len(PLAN_RETRIEVAL)
assert tabla_retrieval["estado"].eq("completo").all()
assert tabla_retrieval["medidas"].eq(tabla_retrieval["total"]).all()
assert np.isfinite(tabla_retrieval["recall@5"]).all()
comparacion = tabla_retrieval.assign(
    aciertos=(tabla_retrieval["recall@5"] * tabla_retrieval["total"]).round().astype(int)
).groupby("configuración").agg(aciertos=("aciertos", "sum"), total=("total", "sum"))
comparacion["recall@5"] = comparacion["aciertos"] / comparacion["total"]
display(comparacion)
comparacion.to_csv(RUTA_RETRIEVAL / "comparacion.csv")
# Contraste comparable de modos: filtros iguales, sin un segundo LLM de reescritura.
candidatas = comparacion.loc[[e for e, (_, filtros, reescritura) in PLAN_RETRIEVAL.items()
                              if filtros and not reescritura]]
SELECCION = candidatas.sort_values("recall@5", ascending=False, kind="stable").index[0]
MODO_RETRIEVAL = PLAN_RETRIEVAL[SELECCION][0]
print("Modo seleccionado para las 40 preguntas:", MODO_RETRIEVAL, "—", SELECCION)
print("El agente elige filtros y consultas; §4 evalúa su comportamiento completo, no filtros oracle.")


Qdrant: 1749 puntos, 1024 dimensiones; colección jchulvi_cloud_92113b6e8fbccc47_e9d702ecfe822c48.
propio 0 · BM25 (control) 14/14


propio 1 · denso plano 14/14


propio 2 · + metadatos 14/14


propio 3 · + híbrido BM25 14/14


propio 4 · reescritura + denso 14/14


...

oficial 4 · reescritura + denso 13/13


oficial 5 · reescritura + híbrido 13/13


,conjunto,configuración,medidas,total,recall@5,reescrituras_por_consulta,estado
0,propio,0 · BM25 (control),14,14,0.285714,0,completo
1,propio,1 · denso plano,14,14,0.642857,0,completo
2,propio,2 · + metadatos,14,14,0.785714,0,completo
3,propio,3 · + híbrido BM25,14,14,0.714286,0,completo
4,propio,4 · reescritura + denso,14,14,0.857143,1,completo
5,propio,5 · reescritura + híbrido,14,14,0.785714,1,completo
6,oficial,0 · BM25 (control),13,13,0.307692,0,completo
7,oficial,1 · denso plano,13,13,0.538462,0,completo
8,oficial,2 · + metadatos,13,13,0.692308,0,completo
9,oficial,3 · + híbrido BM25,13,13,0.538462,0,completo


,aciertos,total,recall@5
configuración,,,
0 · BM25 (control),8,27,0.296296
1 · denso plano,16,27,0.592593
2 · + metadatos,20,27,0.740741
3 · + híbrido BM25,17,27,0.629630
4 · reescritura + denso,22,27,0.814815
5 · reescritura + híbrido,20,27,0.740741


Modo seleccionado para las 40 preguntas: denso — 2 · + metadatos
El agente elige filtros y consultas; §4 evalúa su comportamiento completo, no filtros oracle.


## 4 · Evaluación de las 40 preguntas

Evaluamos por separado las 20 preguntas del equipo y las 20 oficiales: cita, cifra/unidad y herramientas utilizadas.
Se conserva el progreso. Si falla el proveedor, se indica qué queda pendiente.

Progreso por pregunta y traza al terminar, como en S2. Los extractos de herramientas se resumen aquí; §5 conserva el detalle.


In [10]:
# Respuestas independientes de la evaluación. Fuentes efectivas + corpus + vectores:
# cambiar prompt, código compartido o reembebido invalida esta caché.
configuracion_respuestas = {
    "modelo": MODELO_AGENTE, "contexto": "seccion", "modo": MODO_RETRIEVAL,
    "max_llamadas_modelo": agente.MAX_LLAMADAS_MODELO,
    "embeddings": MODELO_EMBEDDINGS if MODO_RETRIEVAL != "bm25" else None,
    "codigo_sha256": HASH_AGENTE,
    "qdrant_sha256": HASH_QDRANT if MODO_RETRIEVAL != "bm25" else None,
    "baseline_sha256": HASH_BASELINE,
    "framework_sha256": {p.relative_to(RAIZ).as_posix(): calcular_sha256(p)
                         for p in sorted((RAIZ / "src/taller_nlp").rglob("*.py"))},
    "corpus": corpus_agente.model_dump(mode="json"),
    "sha256_hijos": corpus.sha256_chunks,
    "indice_cloud": META_CLOUD if MODO_RETRIEVAL != "bm25" else None,
    "coleccion": COLECCION if MODO_RETRIEVAL != "bm25" else None,
}
CACHE_RESPUESTAS = ESTUDIO / ("respuestas_" + huella_json(configuracion_respuestas)[:16])
configuracion_agente = {
    "agente": configuracion_respuestas, "evaluador": configuracion_evaluador,
    "golden_sha256": HASH_GOLDEN,
}
CAMPANA_AGENTE = ESTUDIO / ("campana_" + huella_json(configuracion_agente)[:16])

informes_agente = {}
ESTADOS_EVALUACION = {}

def mostrar_respuesta(respuesta):
    # Formato de pretty_trace de S2, usando la traza real ya registrada.
    print("TRAYECTORIA")
    for paso, llamada in enumerate(respuesta.llamadas, 1):
        args = ", ".join(f"{k}={v!r}" for k, v in llamada.argumentos.items())
        print(f"  {paso}. {llamada.nombre}({args})")
        texto = " ".join((llamada.resultado or "").split())
        print(f"       -> {texto[:220]}{'…' if len(texto) > 220 else ''}")
        if llamada.error:
            print("       ERROR:", llamada.error)
    if not respuesta.llamadas:
        print("  Sin llamadas a herramientas registradas.")
    print("\nRESPUESTA\n" + (respuesta.respuesta or "Sin respuesta."))
    print(f"Fuente: {respuesta.fuente} · Citas: {', '.join(respuesta.citas) or 'sin cita'}")
    coste = f"${respuesta.coste_usd:.6f}" if respuesta.coste_usd is not None else "sin telemetría"
    print(f"Tiempo: {respuesta.latencia_ms / 1000:.1f} s · Coste del agente: {coste}")
    if respuesta.error:
        print("ERROR DEL AGENTE:", respuesta.error)


def evaluar_dataset(dataset):
    manifiesto = CAMPANA_AGENTE / f"{dataset}.manifiesto.json"
    if manifiesto.is_file():
        guardado = ManifiestoExperimento.cargar(manifiesto)
        assert not guardado.comprobar_reproducibilidad(), "No mezclar código nuevo con un informe antiguo."
        ESTADOS_EVALUACION[dataset] = "completo"
        print(f"{dataset}: resultados guardados; sin llamadas nuevas. Detalle completo en §5.")
        por_id = {caso.id: caso for caso in CASOS[dataset]}
        for numero, resultado in enumerate(guardado.informe.resultados, 1):
            print(f"\n[{numero:02d}/20 · {dataset} · {resultado.id_pregunta}] PREGUNTA")
            print(por_id[resultado.id_pregunta].pregunta)
            mostrar_respuesta(resultado.respuesta_agente)
            print("RESULTADO:", "ACIERTO" if resultado.acierto else "FALLO")
            for observacion in resultado.observaciones:
                print("  ->", observacion)
        return guardado.informe
    if MODO_RETRIEVAL != "bm25" and META_CLOUD is None:
        ESTADOS_EVALUACION[dataset] = "pendiente: falta el índice cloud"
        print(dataset, ":", ESTADOS_EVALUACION[dataset])
        return None
    CAMPANA_AGENTE.mkdir(exist_ok=True)
    CACHE_RESPUESTAS.mkdir(exist_ok=True)
    agente.guardar_atomico(CAMPANA_AGENTE / "configuracion.json",
                              json.dumps(configuracion_agente, indent=2).encode())
    try:
        with (qdrant.sesion() if MODO_RETRIEVAL != "bm25" else nullcontext()):
            agente.vector_consulta_cloud.cache_clear()
            constructor = agente.crear_constructor(
                modelo=MODELO_AGENTE, contexto="seccion", modo=MODO_RETRIEVAL,
                modelo_embeddings=MODELO_EMBEDDINGS,
                qdrant_url=qdrant.url if MODO_RETRIEVAL != "bm25" else None,
                coleccion=COLECCION if MODO_RETRIEVAL != "bm25" else None,
                ruta_progreso=CAMPANA_AGENTE / f"{dataset}.progreso.json",
            )
            motor = constructor.construir_motor()
            def responder_y_guardar(pregunta):
                numero, caso = next((i, c) for i, c in enumerate(CASOS[dataset], 1)
                                    if c.pregunta == pregunta)
                print(f"\n[{numero:02d}/20 · {dataset} · {caso.id}] PREGUNTA\n{pregunta}", flush=True)
                ruta = CACHE_RESPUESTAS / (huella_json(pregunta) + ".json")
                if ruta.is_file():
                    print("Respuesta recuperada de caché.", flush=True)
                    respuesta = RespuestaAgente.model_validate_json(ruta.read_text())
                else:
                    print("Generando respuesta y consultando herramientas…", flush=True)
                    respuesta = motor.responder(pregunta)
                    if respuesta.error is None:
                        agente.guardar_atomico(ruta, respuesta.model_dump_json(indent=2).encode())
                mostrar_respuesta(respuesta)
                print("Evaluando cita, cifra y trayectoria; veredicto en §5.", flush=True)
                return respuesta
            informe = constructor.evaluador.evaluar(
                nombre_agente=constructor.nombre, responder=responder_y_guardar, ruta_jsonl=DATASETS[dataset])
            assert informe.numero_preguntas == len(CASOS[dataset]) == 20
            ManifiestoExperimento.desde_constructor(
                constructor, informe=informe, metadatos={**configuracion_agente, "dataset": dataset,
                                                       "etiqueta": "agente_18_llamadas"},
            ).guardar(manifiesto)
            ESTADOS_EVALUACION[dataset] = "completo"
            print(f"\n{dataset}: {informe.numero_preguntas}/20 evaluadas · {informe.aciertos_totales}/20 aciertos.")
            return informe
    except Exception as exc:
        if not es_error_transitorio_modelo(exc):
            raise
        ESTADOS_EVALUACION[dataset] = "pausado: cuota o proveedor; progreso conservado"
        print(dataset, ":", ESTADOS_EVALUACION[dataset])
        return None

In [11]:
# Conjunto creado por el equipo: las 20 preguntas.
informes_agente["propio"] = evaluar_dataset("propio")


[01/20 · propio · gjhh-001] PREGUNTA
¿Qué consecuencias comerciales señala NVIDIA en el Item 1A del ejercicio 2025 si subestima la demanda?


Generando respuesta y consultando herramientas…


TRAYECTORIA
  1. search_filings(query='underestimate demand commercial consequences', ticker='NVDA', fiscal_year=2025, item='1A')
       -> [NVDA-2025-1A-0000] NVDA FY2025 Item 1A (puntuación 0.0164) Item 1A. Risk Factors The following risk factors should be considered in addition to the other information in this Annual Report on Form 10-K. The following ris…

RESPUESTA
NVIDIA señala en el Item 1A del ejercicio 2025 que, si subestima la demanda y sus socios de fundición y fabricantes contratados no pueden aumentar la producción o proporcionar suministro suficiente, podría no satisfacer la mayor demanda de los clientes de forma oportuna o en absoluto. Las consecuencias comerciales serían daño a su reputación y a las relaciones con los clientes, y pérdida de ingresos y cuota de mercado.
Fuente: texto · Citas: NVDA-2025-1A-0000
Tiempo: 19.4 s · Coste del agente: $0.001118
Evaluando cita, cifra y trayectoria; veredicto en §5.



[02/20 · propio · gjhh-002] PREGUNTA
¿De qué recursos físicos depende Microsoft para ampliar y operar sus centros de datos, según el Item 1A, para el ejercicio 2024?


Generando respuesta y consultando herramientas…


...

TRAYECTORIA
  1. list_available()
       -> - **AAPL** — Apple Inc. (CIK 320193): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8] - **MSFT** — MICROSOFT CORP (CIK 789019): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8] - **AMZN** — AMAZON COM INC (CIK 1018724): FY2024 …
  2. get_xbrl_fact(ticker='GOOGL', fiscal_year=2024, concept='OperatingIncomeLoss')
       -> GOOGL FY2024 · OperatingIncomeLoss = 112,390,000,000 USD (cierre de ejercicio 2024-12-31, según el 10-K)
  3. get_xbrl_fact(ticker='GOOGL', fiscal_year=2025, concept='OperatingIncomeLoss')
       -> GOOGL FY2025 · OperatingIncomeLoss = 129,039,000,000 USD (cierre de ejercicio 2025-12-31, según el 10-K)
  4. search_filings(query='operating income increase', ticker='GOOGL', fiscal_year=2025, item='7')
       -> [GOOGL-2025-7-0000] GOOGL FY2025 Item 7 (puntuación 0.0164) ITEM 7.MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS Please read the following discussion and analysis of our financial …

RESP


[20/20 · propio · gjhh-020] PREGUNTA
¿Cómo cambió el flujo de caja de operaciones de Amazon entre FY2024 y FY2025?


Generando respuesta y consultando herramientas…


TRAYECTORIA
  1. list_available()
       -> - **AAPL** — Apple Inc. (CIK 320193): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8] - **MSFT** — MICROSOFT CORP (CIK 789019): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8] - **AMZN** — AMAZON COM INC (CIK 1018724): FY2024 …
  2. get_xbrl_fact(ticker='AMZN', fiscal_year=2024, concept='NetCashProvidedByUsedInOperatingActivities')
       -> AMZN FY2024 · NetCashProvidedByUsedInOperatingActivities = 115,877,000,000 USD (cierre de ejercicio 2024-12-31, según el 10-K)
  3. get_xbrl_fact(ticker='AMZN', fiscal_year=2025, concept='NetCashProvidedByUsedInOperatingActivities')
       -> AMZN FY2025 · NetCashProvidedByUsedInOperatingActivities = 139,514,000,000 USD (cierre de ejercicio 2025-12-31, según el 10-K)
  4. search_filings(query='cash provided by operating activities', ticker='AMZN', fiscal_year=2025, item='7')
       -> [AMZN-2025-7-0000] AMZN FY2025 Item 7 (puntuación 0.0164) Item 7. Management’s Discussion and Analysis of Financial Condition an


propio: 20/20 evaluadas · 19/20 aciertos.


In [12]:
# Conjunto oficial del profesor: las 20 preguntas.
informes_agente["oficial"] = evaluar_dataset("oficial")


[01/20 · oficial · of-001] PREGUNTA
¿Qué dice NVIDIA en su 10-K de FY2025 sobre la competencia en el mercado chino y los controles de exportación?


Generando respuesta y consultando herramientas…


TRAYECTORIA
  1. list_available()
       -> - **AAPL** — Apple Inc. (CIK 320193): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8] - **MSFT** — MICROSOFT CORP (CIK 789019): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8] - **AMZN** — AMAZON COM INC (CIK 1018724): FY2024 …
  2. search_filings(query='competition in China market export controls', ticker='NVDA', fiscal_year=2025, item='7')
       -> [NVDA-2025-7-0000] NVDA FY2025 Item 7 (puntuación 0.0164) Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations The following discussion and analysis of our financial condition and…
  3. search_filings(fiscal_year=2025, item='1A', query='export controls China competition competitive position', ticker='NVDA')
       -> [NVDA-2025-1A-0000] NVDA FY2025 Item 1A (puntuación 0.0164) Item 1A. Risk Factors The following risk factors should be considered in addition to the other information in this Annual Report on Form 10-K. The following ris…

RESPUESTA
No he podido co


[02/20 · oficial · of-002] PREGUNTA
¿Qué riesgo de seguridad asocia Microsoft en FY2025 al uso creciente de modelos de IA generativa en sus sistemas internos?


Generando respuesta y consultando herramientas…


...

TRAYECTORIA
  1. list_available()
       -> - **AAPL** — Apple Inc. (CIK 320193): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8] - **MSFT** — MICROSOFT CORP (CIK 789019): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8] - **AMZN** — AMAZON COM INC (CIK 1018724): FY2024 …
  2. get_xbrl_fact(ticker='AAPL', fiscal_year=2024, concept='GrossProfit')
       -> AAPL FY2024 · GrossProfit = 180,683,000,000 USD (cierre de ejercicio 2024-09-28, según el 10-K)
  3. get_xbrl_fact(ticker='AAPL', fiscal_year=2025, concept='GrossProfit')
       -> AAPL FY2025 · GrossProfit = 195,201,000,000 USD (cierre de ejercicio 2025-09-27, según el 10-K)
  4. search_filings(query='gross margin products', ticker='AAPL', fiscal_year=2025, item='7')
       -> [AAPL-2025-7-0000] AAPL FY2025 Item 7 (puntuación 0.0164) Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read in conjunction with the cons…

RESPUESTA
El beneficio bruto (gross margin) t


[20/20 · oficial · of-020] PREGUNTA
¿Cómo varió el beneficio neto de Meta entre 2024 y 2025, y qué explica esa variación?


Generando respuesta y consultando herramientas…


TRAYECTORIA
  1. list_available()
       -> - **AAPL** — Apple Inc. (CIK 320193): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8] - **MSFT** — MICROSOFT CORP (CIK 789019): FY2024 [1A, 7, 7A, 8]; FY2025 [1A, 7, 7A, 8] - **AMZN** — AMAZON COM INC (CIK 1018724): FY2024 …
  2. get_xbrl_fact(ticker='META', fiscal_year=2024, concept='NetIncomeLoss')
       -> META FY2024 · NetIncomeLoss = 62,360,000,000 USD (cierre de ejercicio 2024-12-31, según el 10-K)
  3. get_xbrl_fact(ticker='META', fiscal_year=2025, concept='NetIncomeLoss')
       -> META FY2025 · NetIncomeLoss = 60,458,000,000 USD (cierre de ejercicio 2025-12-31, según el 10-K)
  4. search_filings(query='net income decrease 2025 compared to 2024', ticker='META', fiscal_year=2025, item='7')
       -> [META-2025-7-0000] META FY2025 Item 7 (puntuación 0.0164) Item 7.Management's Discussion and Analysis of Financial Condition and Results of Operations You should read the following discussion of our financial condition a…

RESPUESTA
El benefi


oficial: 20/20 evaluadas · 17/20 aciertos.


## 5 · Resultados

Solo resultados del protocolo actual. Las notas anteriores no son comparables: [historial](README.md#resultados-historicos).
Mostramos las 40 preguntas con respuesta completa, referencia, criterios, citas y trazas. Los pendientes quedan sin nota.


In [13]:
informes = {}
for dataset, informe in informes_agente.items():
    informes[(dataset, "Agente 18 llamadas (actual)")] = informe

def fila_resumen(dataset, version, informe):
    fila = {"conjunto": dataset, "versión": version,
            "estado": "completo" if informe is not None else ESTADOS_EVALUACION.get(dataset, "pendiente")}
    if informe is None:
        return fila
    resultados = informe.resultados
    def media(valores):
        validos = [v for v in valores if v is not None]
        return float(np.mean(validos)) if validos else None
    return {**fila, "protocolo": informe.version_protocolo_citas,
            "preguntas": informe.numero_preguntas, "aciertos": informe.aciertos_totales,
            "cita": media([bool(r.cita_existe and r.cita_respalda) for r in resultados if r.cita_existe is not None]),
            "cifra": media([r.cifra_correcta for r in resultados]),
            "trayectoria": media([r.trayectoria_correcta for r in resultados]),
            "recall@5": informe.recall_at_k_medio,
            "coste medio agente (¢)": informe.coste_medio_usd * 100 if informe.coste_medio_usd is not None else None,
            "latencia media (s)": informe.latencia_media_ms / 1000,
            "llamadas/pregunta": informe.llamadas_por_pregunta,
            **{f"aciertos {f}": f"{informe.aciertos_por_familia[f]}/{informe.preguntas_por_familia[f]}"
               for f in ("extractiva", "numerica", "comparativa")}}
resumen = pd.DataFrame([fila_resumen(d, v, i) for (d, v), i in informes.items()])
display(resumen.round(3))
CAMPANA_AGENTE.mkdir(exist_ok=True)
resumen.to_csv(CAMPANA_AGENTE / "resumen.csv", index=False)
assert set(resumen["conjunto"]) == {"propio", "oficial"}
print("§5: ninguna celda pendiente se convierte en una nota cero.")

,conjunto,versión,estado,protocolo,preguntas,aciertos,cita,cifra,trayectoria,recall@5,coste medio agente (¢),latencia media (s),llamadas/pregunta,aciertos extractiva,aciertos numerica,aciertos comparativa
0,propio,Agente 18 llamadas (actual),completo,5,20,19,0.929,0.929,1.0,0.786,0.133,28.232,3.45,6/6,6/6,7/8
1,oficial,Agente 18 llamadas (actual),completo,5,20,17,0.769,0.929,1.0,1.000,0.213,86.700,3.45,4/6,7/7,6/7


§5: ninguna celda pendiente se convierte en una nota cero.


In [14]:
# Los 40 casos, incluidos los pendientes; esta celda no llama a modelos.
detalles = {}
for dataset, casos in GOLDEN.items():
    informe = informes_agente.get(dataset)
    resultados = list(informe.resultados) if informe is not None else []
    progreso = CAMPANA_AGENTE / f"{dataset}.progreso.json"
    if informe is None and progreso.is_file():
        guardado = json.loads(progreso.read_text())
        assert guardado["contexto"]["sha256_golden"] == HASH_GOLDEN[dataset]
        assert guardado["contexto"]["ids_casos"] == [g["id"] for g in casos]
        assert huella_json(json.loads((CAMPANA_AGENTE / "configuracion.json").read_text())) == huella_json(configuracion_agente)
        resultados = [ResultadoPregunta.model_validate(r) for r in guardado["resultados"]]
    por_id = {r.id_pregunta: r for r in resultados}
    assert len(por_id) == len(resultados) and set(por_id) <= {g["id"] for g in casos}
    filas = []
    for g in casos:
        r = por_id.get(g["id"])
        respuesta = r.respuesta_agente if r is not None else None
        cache = CACHE_RESPUESTAS / (huella_json(g["pregunta"]) + ".json")
        if respuesta is None and cache.is_file():
            respuesta = RespuestaAgente.model_validate_json(cache.read_text())
        estado = ("acierto" if r.acierto else "fallo") if r is not None else (
            "respuesta guardada; evaluación pendiente" if respuesta is not None
            else ESTADOS_EVALUACION.get(dataset, "pendiente"))
        if r is not None and respuesta.error:
            estado = "fallo técnico del agente"
        filas.append({
            "conjunto": dataset, "id": g["id"], "familia": g["familia"],
            "pregunta": g["pregunta"], "referencia_esperada": g["respuesta_esperada"],
            "respuesta": respuesta.respuesta if respuesta else None, "estado": estado,
            "acierto": r.acierto if r else None,
            "cita_existe": r.cita_existe if r else None,
            "cita_respalda": r.cita_respalda if r else None,
            "cifra_correcta": r.cifra_correcta if r else None,
            "trayectoria_correcta": r.trayectoria_correcta if r else None,
            "cifra_esperada": g["cifra_esperada"], "unidad_esperada": g["unidad"],
            "cifra_agente": respuesta.cifra if respuesta else None,
            "unidad_agente": respuesta.unidad if respuesta else None,
            "fuente": respuesta.fuente if respuesta else None,
            "cita": respuesta.cita if respuesta else None,
            "chunk_ids": list(respuesta.citas) if respuesta else [],
            "observaciones": list(r.observaciones) if r else [],
            "error": respuesta.error if respuesta else None,
            "latencia_ms": respuesta.latencia_ms if respuesta else None,
            "coste_usd": respuesta.coste_usd if respuesta else None,
            "herramientas_esperadas": g["herramienta_esperada"],
            "traza": [t.model_dump(mode="json") for t in respuesta.llamadas] if respuesta else [],
        })
    assert len(filas) == 20
    detalles[dataset] = pd.DataFrame(filas)
    detalles[dataset].to_csv(CAMPANA_AGENTE / f"{dataset}.detalle.csv", index=False)
    agente.guardar_atomico(CAMPANA_AGENTE / f"{dataset}.detalle.json",
                              json.dumps(filas, ensure_ascii=False, indent=2).encode())
    contestadas = sum(bool(f["respuesta"]) and f["error"] is None for f in filas)
    print(f"{dataset}: {len(resultados)}/20 evaluadas; {contestadas}/20 respuestas sin error del agente.")
    with pd.option_context("display.max_rows", None):
        display(detalles[dataset][["id", "familia", "estado", "cita_existe",
                                  "cita_respalda", "cifra_correcta", "trayectoria_correcta"]])
    for fila in filas:
        print(f"\n{fila['id']} · {fila['estado']}\n{fila['pregunta']}")
        for campo, valor in fila.items():
            if campo in {"conjunto", "id", "estado", "pregunta"}:
                continue
            if isinstance(valor, (list, dict)):
                valor = json.dumps(valor, ensure_ascii=False, indent=2)
            print(f"{campo.replace('_', ' ')}: {valor if valor is not None else 'No disponible / no aplica'}")
assert sum(len(d) for d in detalles.values()) == 40
procesadas = sum(int(d["acierto"].notna().sum()) for d in detalles.values())
print(f"Evaluadas: {procesadas}/40. Pendientes: {40 - procesadas}/40.")
print("40 casos visibles y exportados. Pendiente no significa fallo; respuesta no significa acierto.")


propio: 20/20 evaluadas; 19/20 respuestas sin error del agente.


,id,familia,estado,cita_existe,cita_respalda,cifra_correcta,trayectoria_correcta
0,gjhh-001,extractiva,acierto,True,True,None,True
1,gjhh-002,extractiva,acierto,True,True,None,True
2,gjhh-003,extractiva,acierto,True,True,None,True
3,gjhh-004,extractiva,acierto,True,True,None,True
4,gjhh-005,extractiva,acierto,True,True,None,True
5,gjhh-006,extractiva,acierto,True,True,None,True
6,gjhh-007,numerica,acierto,None,None,True,True
7,gjhh-008,numerica,acierto,None,None,True,True
8,gjhh-009,numerica,acierto,None,None,True,True
9,gjhh-010,numerica,acierto,None,None,True,True



gjhh-001 · acierto
¿Qué consecuencias comerciales señala NVIDIA en el Item 1A del ejercicio 2025 si subestima la demanda?
familia: extractiva
referencia esperada: Podría no atender a tiempo el aumento de la demanda; además, podría dañar su reputación y sus relaciones con clientes, y perder ingresos y cuota de mercado.
respuesta: NVIDIA señala en el Item 1A del ejercicio 2025 que, si subestima la demanda y sus socios de fundición y fabricantes contratados no pueden aumentar la producción o proporcionar suministro suficiente, podría no satisfacer la mayor demanda de los clientes de forma oportuna o en absoluto. Las consecuencias comerciales serían daño a su reputación y a las relaciones con los clientes, y pérdida de ingresos y cuota de mercado.
acierto: True
cita existe: True
cita respalda: True
cifra correcta: No disponible / no aplica
trayectoria correcta: True
cifra esperada: No disponible / no aplica
unidad esperada: No disponible / no aplica
cifra agente: No disponible / no aplica

,id,familia,estado,cita_existe,cita_respalda,cifra_correcta,trayectoria_correcta
0,of-001,extractiva,fallo técnico del agente,False,False,None,True
1,of-002,extractiva,acierto,True,True,None,True
2,of-003,extractiva,acierto,True,True,None,True
3,of-004,extractiva,fallo técnico del agente,False,False,None,True
4,of-005,extractiva,acierto,True,True,None,True
5,of-006,extractiva,acierto,True,True,None,True
6,of-007,numerica,acierto,None,None,True,True
7,of-008,numerica,acierto,None,None,True,True
8,of-009,numerica,acierto,None,None,True,True
9,of-010,numerica,acierto,None,None,True,True



of-001 · fallo técnico del agente
¿Qué dice NVIDIA en su 10-K de FY2025 sobre la competencia en el mercado chino y los controles de exportación?
familia: extractiva
referencia esperada: Que el mercado chino, donde sus productos están limitados por los controles de exportación, es muy competitivo y espera que lo siga siendo.
respuesta: No he podido completar una respuesta estructurada y verificada dentro de los límites de ejecución. No presento datos ni citas sin validar. La ejecución queda marcada como fallida.
acierto: False
cita existe: False
cita respalda: False
cifra correcta: No disponible / no aplica
trayectoria correcta: True
cifra esperada: No disponible / no aplica
unidad esperada: No disponible / no aplica
cifra agente: No disponible / no aplica
unidad agente: No disponible / no aplica
fuente: ninguna
cita: No disponible / no aplica
chunk ids: []
observaciones: [
  "Error del agente: El agente terminó sin una respuesta estructurada.",
  "Falta una cita o algún chunk_id no ex

## 6 · Entrega

La interfaz compartida expone `responder()` y `evaluar()`. El hold-out se ejecuta aparte:

```bash
python -m agente --variante experimentos.jchulvi.agente --evaluar holdout.jsonl --salida resultados/holdout.csv
```


In [15]:
# Selector requerido por la interfaz compartida de entrega, no por la evaluación.
os.environ["TALLER_VARIANTE_AGENTE"] = "experimentos.jchulvi.agente"
from agente.interfaz import responder, evaluar
assert callable(responder) and callable(evaluar)
print("Interfaz seleccionada: agente.py. Importar no carga modelos ni ejecuta preguntas.")
print("Resultados actuales:", CAMPANA_AGENTE.relative_to(ESTUDIO))
print("Resultados retrieval:", RUTA_RETRIEVAL.relative_to(ESTUDIO))
cuenta_final = leer_cuenta()
gasto = {"inicio": CUENTA_INICIAL, "fin": cuenta_final,
         "variacion_uso_cuenta_usd": cuenta_final["usage"] - CUENTA_INICIAL["usage"],
         "nota": "Solo este arranque: excluye gasto previo si se reanuda y puede incluir otros usos simultáneos de la clave."}
sello = CUENTA_INICIAL["utc"].replace(":", "-")
agente.guardar_atomico(CAMPANA_AGENTE / f"gasto_{sello}.json",
                          json.dumps(gasto, indent=2).encode())
print("Variación de uso de la cuenta en esta ejecución (USD):",
      round(gasto["variacion_uso_cuenta_usd"], 6))


Interfaz seleccionada: agente.py. Importar no carga modelos ni ejecuta preguntas.
Resultados actuales: campana_0d884de76bd86303
Resultados retrieval: retrieval_s2_1f6c6eef2429b491
Variación de uso de la cuenta en esta ejecución (USD): 0.070382
